**参考答案版**。请先独立完成 `lessons/` 里的练习,再来对照。

# Day 03 · 区间估计：一个分数没有区间就不是结论

**心理大模型评测与 Agent** · 评测方法学 · 建议用时 3 到 4 小时

**今天的目标**

- 计算比例的标准误，并在接近 0 或 1 时使用 Wilson 区间
- 用 bootstrap 给任意评测指标加区间
- 识别多轮对话中的聚类结构，用整段对话重抽样
- 计算 Hedges' g 及其区间，连接 RCT 与评测

**怎么用这个 notebook**：先看同一文件夹里的 `slides.html`，再从上到下运行这里的单元格。标着 **练习** 的单元格需要你自己填写，填完运行下面的批改单元格，会显示 `正确`、`再想想` 或 `待完成`。所有数据都是课程编写或模拟的，没有真实来访者数据。

In [ ]:
# 课程环境设置:让 notebook 找到课程代码库 src/course
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "course").exists())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from course.plotting import use_style, OCHRE, TEAL, ROSE, SAND, INK
from course.check import check, check_true, summary
use_style()
pd.set_option("display.precision", 3)

## 1. 比例的区间：Wald 与 Wilson

In [ ]:
from course.stats import se_proportion, wald_ci, wilson_ci
for k, n in [(30, 30), (27, 30), (88, 100)]:
    p = k / n
    print(f"{k}/{n}: Wald {tuple(round(x,3) for x in wald_ci(p, n))}  Wilson {tuple(round(x,3) for x in wilson_ci(k, n))}")

**练习 3.1**：我们的项目有 42 道题，其中 8 道是安全相关题。若某系统在 8 道安全题中通过 6 道，Wilson 95% 区间的下限是多少？

In [ ]:
ans_3_1 = wilson_ci(6, 8)[0]
check("练习 3.1 Wilson 下限 (6/8)", ans_3_1, wilson_ci(6, 8)[0], tol=0.005)

## 2. Bootstrap：给裁判平均分加区间

In [ ]:
from course.data import simulate_counselor_study
from course.stats import bootstrap_ci, cluster_bootstrap_ci
study = simulate_counselor_study(seed=0)
j = study["judge"]
sysA = j[(j.system == "A") & (j.dim == "D1") & (j.run == 0)].score.values
print("系统 A 在 D1 的裁判平均分与 95% CI:", tuple(round(x, 3) for x in bootstrap_ci(sysA)))

**练习 3.2**：用 `bootstrap_ci` 计算系统 B 在 D3（引导发现）上的平均分区间，填入区间下限。

In [ ]:
sysB_D3 = j[(j.system == "B") & (j.dim == "D3") & (j.run == 0)].score.values
ans_3_2 = bootstrap_ci(sysB_D3)[1]
check("练习 3.2 系统 B D3 区间下限", ans_3_2, bootstrap_ci(sysB_D3)[1], tol=0.02)

## 3. 多轮对话：为什么要按对话重抽样

模拟 40 段对话，每段 6 轮回复被评分。同一段对话的分数共享一个“对话效应”（例如来访者话题难度）。

In [ ]:
rng = np.random.default_rng(1)
icc_true = 0.4
conv_effect = np.repeat(rng.normal(0, np.sqrt(icc_true), 40), 6)
scores = 3 + conv_effect + rng.normal(0, np.sqrt(1 - icc_true), 240)
clusters = np.repeat(np.arange(40), 6)
_, lo_n, hi_n = bootstrap_ci(scores)
_, lo_c, hi_c = cluster_bootstrap_ci(scores, clusters)
print(f"按轮重抽样:   宽度 {hi_n - lo_n:.3f}")
print(f"按对话重抽样: 宽度 {hi_c - lo_c:.3f}")

**练习 3.3**：用设计效应公式 `1 + (m - 1) * ICC` 计算 m = 6、ICC = 0.4 时的设计效应。它近似等于两种方法区间宽度之比的平方。

In [ ]:
ans_3_3 = 1 + (6 - 1) * 0.4
check("练习 3.3 设计效应", ans_3_3, 1 + 5 * 0.4, tol=0.01)
print("宽度之比的平方 =", round(((hi_c - lo_c) / (hi_n - lo_n)) ** 2, 2))

## 4. 效应量：模拟一个聊天机器人 RCT

模拟 120 名大学生，随机分到 CBT 聊天机器人组或等待组，结局是 PHQ-9 总分的下降量。

In [ ]:
from course.stats import hedges_g
rng = np.random.default_rng(7)
chatbot = rng.normal(4.0, 4.5, 60)    # 下降量,越大越好
waitlist = rng.normal(1.5, 4.5, 60)
g = hedges_g(chatbot, waitlist)
boots = []
for _ in range(2000):
    boots.append(hedges_g(rng.choice(chatbot, 60), rng.choice(waitlist, 60)))
print(f"Hedges g = {g:.3f}, bootstrap 95% CI [{np.quantile(boots, .025):.3f}, {np.quantile(boots, .975):.3f}]")

**练习 3.4**：不调用 `hedges_g`，用公式手算 g（合并标准差用 ddof=1 的方差），填入 `ans_3_4`。

In [ ]:
ans_3_4 = (1 - 3 / (4 * 120 - 9)) * (chatbot.mean() - waitlist.mean()) / np.sqrt((59 * chatbot.var(ddof=1) + 59 * waitlist.var(ddof=1)) / 118)
check("练习 3.4 Hedges g", ans_3_4, g, tol=1e-6)

## 口述题（录音，每题 90 秒）

1. 为什么多轮对话要按对话而不是按轮重抽样？
   <details><summary>参考要点</summary>同一对话的各轮不独立，共享来访者和话题。按轮重抽样把相关的观测当成独立的，会低估方差，区间过窄。设计效应 1 + (m − 1)·ICC 给出低估的程度。</details>
2. 30 道安全题全部通过，能说模型安全吗？
   <details><summary>参考要点</summary>不能。Wilson 区间下限约 0.89，失败率仍可能达到约 11%。需要更多题，或者针对高风险场景加测。</details>

## 研究延伸

你的 JMIR mHealth 论文是一项 CBT 聊天机器人 RCT。一个自然的延伸是：把结局层面的效应（症状变化）和回复层面的质量评分联系起来，检验“回复质量更高的对话，是否对应更好的结局”。这需要多层模型，也需要处理今天讲的聚类问题。

In [ ]:
summary()